# 03 — Push the reversible model to the maximum batch size

The paper's headline (Table 3): with the same GPU, a reversible model fits ~10× the batch of the baseline, and throughput goes up because
GPUs are more efficient on bigger batches (Table 4). Here we:

1. **Probe** the largest batch that survives a full forward+backward+optimizer step, for the baseline and for the reversible model
   (doubling, then binary search).
2. **Train** the reversible model for the same 50M tokens at (≈90% of) its max batch, with the learning rate scaled by √(B/32).
3. Compare loss / tokens-per-second / peak memory against the fixed-batch runs.

> One thing to know before reading the numbers: with only 20M parameters and a 50k-word vocabulary, the *output layer's logits*
> (`batch × 256 × 50257` numbers) are a large share of memory — and reversibility does nothing about them. We use a chunked,
> checkpointed LM head for **all** models (baseline included) so that this doesn't hide the effect of reversibility on the transformer stack.

In [ ]:
# @title Setup — clone repo (if needed), install deps, detect GPU
import os, sys, subprocess, json, time, math
REPO_URL = "https://github.com/swatibansal/reversible-llm-poc.git"

if not os.path.exists("src/revllm.py"):
    if os.path.exists("../src/revllm.py"):
        os.chdir("..")
    else:
        subprocess.run(["git", "clone", "-q", REPO_URL, "reversible-llm-poc"], check=True)
        os.chdir("reversible-llm-poc")
sys.path.insert(0, os.path.abspath("src"))
subprocess.run([sys.executable, "-m", "pip", "-q", "install", "tiktoken", "datasets", "matplotlib"], check=False)

import torch
from revllm import Config, GPT, SavedTensorMeter
from data import prepare_tinystories, prepare_synthetic, TokenStream
import train as T

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
# SMOKE mode = tiny synthetic run that finishes in ~1 min on CPU. Auto-enabled when there is no GPU.
SMOKE = os.environ.get("SMOKE", "0") == "1" or DEVICE == "cpu"
print("device:", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "cpu", "| SMOKE mode:", SMOKE)
os.makedirs("results", exist_ok=True)


In [ ]:
# @title Persist results/ and data/ to Google Drive (Colab only)
# Surviving session resets matters twice here: results/*.json are the deliverable,
# and the tokenized TinyStories memmaps take ~4 min to rebuild from scratch.
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB:
    drive.mount("/content/drive")
    for name in ("results", "data"):
        target = f"/content/drive/MyDrive/revllm/{name}"
        os.makedirs(target, exist_ok=True)
        os.system(f"rm -rf {name} && ln -s {target} {name}")
    print("results/ and data/ now live on Drive (MyDrive/revllm/)")
else:
    print("not running on Colab - keeping local results/ and data/")


In [ ]:
# @title Experiment configuration (shared by all notebooks)
if SMOKE:
    MODEL  = dict(vocab_size=512, block_size=64, n_layer=4, n_embd=128, n_head=4)
    TOKENS = 200_000          # token budget
    BATCH  = 16               # the fixed batch size for notebooks 01/02
    LR     = 2e-3
    LOG    = dict(eval_every=100, eval_iters=5, log_every=50)
else:
    # ~20.8M parameters (12.9M in the tied GPT-2 embedding, 7.9M in 10 transformer blocks of width 256)
    MODEL  = dict(vocab_size=50257, block_size=256, n_layer=10, n_embd=256, n_head=4)
    TOKENS = 50_000_000
    BATCH  = 32               # 32 x 256 = 8,192 tokens / step  -> ~6,100 steps for 50M tokens
    LR     = 6e-4
    LOG    = dict(eval_every=250, eval_iters=20, log_every=50)

def get_data():
    if SMOKE:
        return prepare_synthetic("data/synthetic", 2_000_000, 200_000, vocab=MODEL["vocab_size"])
    return prepare_tinystories("data/tinystories", n_train_tokens=55_000_000, n_val_tokens=2_000_000)

def gpu_table(rows, headers):
    w = [max(len(str(r[i])) for r in [headers] + rows) for i in range(len(headers))]
    line = lambda r: "| " + " | ".join(str(c).ljust(w[i]) for i, c in enumerate(r)) + " |"
    print(line(headers)); print("|" + "|".join("-" * (x + 2) for x in w) + "|")
    for r in rows: print(line(r))

import matplotlib.pyplot as plt
def plot_runs(results, key="curve", title="training loss", smooth=25):
    plt.figure(figsize=(8, 4.5))
    for r in results:
        c = r[key]
        if not c: continue
        xs = [p[1] / 1e6 for p in c]; ys = [p[2] for p in c]
        if key == "curve" and smooth > 1 and len(ys) > smooth:
            ys = [sum(ys[max(0, i - smooth):i + 1]) / len(ys[max(0, i - smooth):i + 1]) for i in range(len(ys))]
        plt.plot(xs, ys, label=f"{r['run_name']}  (final {ys[-1]:.3f})")
    plt.xlabel("tokens seen (M)"); plt.ylabel("cross-entropy (nats)"); plt.title(title); plt.legend(); plt.grid(alpha=.3)
    plt.show()


## 1 — maximum batch that fits

In [ ]:
BEST = "midpoint"        # pick the variant that did best in notebook 02
H = {"midpoint": 0.5, "leapfrog": 1.0, "hamiltonian": 1.0}
probe = {}
if DEVICE == "cuda":
    for name, cfg in [("baseline", Config(mode="baseline", **MODEL)),
                      (f"{BEST}/no-rev", Config(mode=BEST, h=H[BEST], rev_backprop=False, **MODEL)),
                      (f"{BEST}/rev", Config(mode=BEST, h=H[BEST], rev_backprop=True, **MODEL))]:
        print(name)
        b, pk = T.find_max_batch(cfg, start=16)
        probe[name] = {"max_batch": b, "peak_gb": pk}
    gpu_table([[k, v["max_batch"], f"{v['peak_gb']:.2f}", f"{v['max_batch']/probe['baseline']['max_batch']:.1f}x"] for k, v in probe.items()],
              ["model", "max batch (x%d tokens)" % MODEL["block_size"], "peak GB at max", "vs baseline"])
    json.dump({"gpu": torch.cuda.get_device_name(0), "total_gb": torch.cuda.get_device_properties(0).total_memory/1e9, "probe": probe},
              open("results/max_batch_probe.json", "w"), indent=1)
    MAXB = int(probe[f"{BEST}/rev"]["max_batch"] * 0.9) // 8 * 8      # 10% headroom for fragmentation
else:
    print("no GPU — SMOKE: pretending max batch is 64"); MAXB = 64
print("training batch for the max-batch run:", MAXB)

## 2 — train the reversible model at max batch, same 50M-token budget

Fewer, bigger steps. LR is scaled by √(B/32) (square-root scaling rule) and capped at 3e-3.

In [ ]:
train_bin, val_bin = get_data()
lr_big = min(LR * math.sqrt(MAXB / BATCH), 3e-3)
cfg = Config(mode=BEST, h=H[BEST], rev_backprop=True, **MODEL)
res_big, model = T.train(cfg, train_bin, val_bin, batch_size=MAXB, tokens_budget=TOKENS, lr=lr_big,
                         out_json=f"results/{BEST}_maxbatch_B{MAXB}.json",
                         eval_every=max(10, LOG["eval_every"] * BATCH // MAXB), eval_iters=LOG["eval_iters"], log_every=max(1, 50 * BATCH // MAXB))

### Optional: baseline at *its* max batch too (fair "each model at its own limit" comparison)
Set `ALSO_BASELINE_MAX = True` to run it (another ~25–40 min on T4).

In [ ]:
ALSO_BASELINE_MAX = False
if ALSO_BASELINE_MAX and DEVICE == "cuda":
    Bb = int(probe["baseline"]["max_batch"] * 0.9) // 8 * 8
    cfgb = Config(mode="baseline", **MODEL)
    T.train(cfgb, train_bin, val_bin, batch_size=Bb, tokens_budget=TOKENS, lr=min(LR * math.sqrt(Bb / BATCH), 3e-3),
            out_json=f"results/baseline_maxbatch_B{Bb}.json",
            eval_every=max(10, LOG["eval_every"] * BATCH // Bb), eval_iters=LOG["eval_iters"], log_every=max(1, 50 * BATCH // Bb))

## 3 — everything side by side

In [ ]:
allres = [json.load(open("results/" + f)) for f in sorted(os.listdir("results")) if f.endswith(".json")]
allres = [r for r in allres if "curve" in r and not r["run_name"].endswith("_short")]
rows = [[r["run_name"], r["batch_size"], r["steps"], f"{r['final_train_loss']:.4f}", f"{r['final_val_loss']:.4f}",
         f"{r['tokens_per_s_steady']:,.0f}", f"{r['peak_mem_gb']:.2f}" if r["peak_mem_gb"] else "n/a", f"{r['wall_time_s']/60:.1f}"] for r in allres]
gpu_table(rows, ["run", "batch", "steps", "train loss", "val loss", "tokens/s", "peak GB", "minutes"])
plot_runs(allres, title="training loss — all runs (x axis = tokens, so batch sizes are comparable)")
plot_runs(allres, key="val_curve", title="validation loss — all runs")
plt.savefig("results/loss_curves_all.png", dpi=120)